In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys


def find_hw02_root():
    cwd = Path.cwd().resolve()
    for path in (cwd, *cwd.parents):
        if (path / 'code' / 'deblur_functions.ipynb').exists() and (path / 'data').exists():
            return Path(os.path.relpath(path, cwd))

        hw02 = path / 'hw02'
        if (hw02 / 'code' / 'deblur_functions.ipynb').exists() and (hw02 / 'data').exists():
            return Path(os.path.relpath(hw02, cwd))

    raise FileNotFoundError('Run this notebook from the repository root, hw02, or hw02/code.')


if 'FOLDER_NAME' not in globals():
    FOLDER_NAME = find_hw02_root()

if importlib.util.find_spec('proximal') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'proximal==0.1.7'])


In [ ]:
%run "{FOLDER_NAME}/code/deblur_functions.ipynb"

In [ ]:
''' Solve deblur problem by Proximal '''
import sys

import scipy.misc
import scipy.datasets
scipy.misc.ascent = scipy.datasets.ascent
from proximal.utils.utils import *
from proximal.halide.halide import *
from proximal.lin_ops import *
from proximal.prox_fns import *
from proximal.algorithms import *

import numpy as np
import time


In [ ]:
def TVL1(img_in, k_in, max_iter, lamb_da):
    """ TVL1 deblur
            Args:
                img_in (uint8 ndarray, shape(height, width, ch)): Blurred image
                k_in (uint8 ndarray, shape(height, width)): Blur kernel
                max_iter (int): Total iteration count
                lamb_da (float): TVL1 variable

            Returns:
                TVL1_result (uint8 ndarray, shape(height, width, ch)): Deblurred image
    """

    img = img_conversion(img_in, to_float = 1)
    K = kernel_conversion(k_in)

    K_rgb = np.zeros((K.shape[0], K.shape[1], 3))
    K_rgb[:,:,0] = K
    K_rgb[:,:,1] = K
    K_rgb[:,:,2] = K
    K = K_rgb

    # test the solver with some sparse gradient deconvolution
    eps_abs_rel = 1e-3
    test_solver = 'pc'

    #%% rgb channels
    TVL1_result = Variable(img.shape)

    # model the problem by proximal
    prob = Problem(norm1(conv(K, TVL1_result, dims = 2) - img) + lamb_da * group_norm1(grad(TVL1_result, dims = 2), [3] ) + nonneg(TVL1_result)) # formulate problem

    # solve the problem
    result = prob.solve(verbose = False, solver = test_solver, x0 = img, eps_abs = eps_abs_rel, eps_rel = eps_abs_rel, max_iters = max_iter) # solve problem
    TVL1_result = TVL1_result.value

    # output color image
    TVL1_result = img_conversion(TVL1_result, to_float = 0)
    return TVL1_result

In [ ]:
def TVL2(img_in, k_in, max_iter, lamb_da):
    """ TVL2 deblur
            Args:
                img_in (uint8 ndarray, shape(height, width, ch)): Blurred image
                k_in (uint8 ndarray, shape(height, width)): blur kernel
                max_iter (int): total iteration count
                lamb_da (float): TVL2 variable

            Returns:
                TVL2_result (uint8 ndarray, shape(height, width, ch)): deblurred image

            Todo:
                TVL2 deblur (Note that you shall be calling img_conversion and kernel_conversion)
    """

    ''' TODO '''

    img = img_conversion(img_in, to_float = 1)
    K = kernel_conversion(k_in)

    K_rgb = np.zeros((K.shape[0], K.shape[1], 3))
    K_rgb[:,:,0] = K
    K_rgb[:,:,1] = K
    K_rgb[:,:,2] = K
    K = K_rgb

    # test the solver with some sparse gradient deconvolution
    eps_abs_rel = 1e-3
    test_solver = 'pc'

    #%% rgb channels
    TVL2_result = Variable(img.shape)

    # model the problem by proximal
    prob = Problem(sum_squares(conv(K, TVL2_result, dims = 2) - img) + lamb_da * group_norm1(grad(TVL2_result, dims = 2), [3]) + nonneg(TVL2_result)) # formulate problem

    # solve the problem
    result = prob.solve(verbose = False, solver = test_solver, x0 = img, eps_abs = eps_abs_rel, eps_rel = eps_abs_rel, max_iters = max_iter) # solve problem
    TVL2_result = TVL2_result.value

    # output color image
    TVL2_result = img_conversion(TVL2_result, to_float = 0)

    return TVL2_result


In [ ]:
def TVpoisson(img_in, k_in, max_iter, lamb_da):
    """ TVLpoisson deblur
            Args:
                img_in (uint8 ndarray, shape(height, width, ch)): Blurred image
                k_in (uint8 ndarray, shape(height, width)): blur kernel
                max_iter (int): total iteration count
                lamb_da (float): TVpoisson variable

            Returns:
                TVpoisson_result (uint8 ndarray, shape(height, width, ch)): deblurred image

            Todo:
                TVpoisson deblur (Note that you shall be calling img_conversion and kernel_conversion)
    """

    ''' TODO '''

    img = img_conversion(img_in, to_float = 1)
    K = kernel_conversion(k_in)

    K_rgb = np.zeros((K.shape[0], K.shape[1], 3))
    K_rgb[:,:,0] = K
    K_rgb[:,:,1] = K
    K_rgb[:,:,2] = K
    K = K_rgb

    # test the solver with some sparse gradient deconvolution
    eps_abs_rel = 1e-3
    test_solver = 'pc'

    #%% rgb channels
    TVpoisson_result = Variable(img.shape)

    # model the problem by proximal
    prob = Problem(poisson_norm(conv(K, TVpoisson_result, dims = 2), img) + lamb_da * group_norm1(grad(TVpoisson_result, dims = 2), [3]) + nonneg(TVpoisson_result)) # formulate problem

    # solve the problem
    result = prob.solve(verbose = False, solver = test_solver, x0 = img, eps_abs = eps_abs_rel, eps_rel = eps_abs_rel, max_iters = max_iter) # solve problem
    TVpoisson_result = TVpoisson_result.value

    # output color image
    TVpoisson_result = img_conversion(TVpoisson_result, to_float = 0)

    return TVpoisson_result